# 處理美股資料

1. 在 網址： https://www.nasdaq.com/market-activity/stocks/brk.b/historical?page=1&rows_per_page=10&timeline=y5
  下載歷史資料
2. 放到 "history_data/us/stock_price_raw"


# 讀取舊格式的檔案並轉換為新格式

In [9]:
import csv
import os
from datetime import datetime


def convert_file(input_file, output_file):
    with open(input_file, mode='r', encoding='utf-8') as infile:
        reader = csv.DictReader(infile)
        rows = []

        for row in reader:

            date = datetime.strptime(row["Date"], "%m/%d/%Y").strftime("%Y%m%d")

            # replace $ sign and comma
            close_price = row["Close/Last"].replace("$", "")
            open_price = row["Open"].replace("$", "")
            high_price = row["High"].replace("$", "")
            low_price = row["Low"].replace("$", "")
            volume = row["Volume"].replace(",", "")

            rows.append({
                "Date": date,
                "Volume": volume,
                "Open": open_price,
                "High": high_price,
                "Low": low_price,
                "Close": close_price,
            })
    # reverse
    rows = rows[::-1]
    
    os.makedirs(os.path.dirname(output_file), exist_ok=True)
    with open(output_file, mode='w', encoding='utf-8', newline='') as outfile:
        fieldnames = ["Date", "Volume",  "Open", "High", "Low", "Close"]
        writer = csv.DictWriter(outfile, fieldnames=fieldnames)

        writer.writeheader()
        writer.writerows(rows)



# 畫技術分析


In [10]:
import pandas as pd
import numpy as np
import json
import requests
import time
import os

# 波動率
def historical_volatility(prices, window=10, trading_days=252):
    """
    計算歷史波動率 (Historical Volatility)

    :param prices: 一個包含價格數據的列表或Numpy數組
    :param window: 計算波動率的視窗期 (默認是252天)
    :param trading_days: 年化的交易天數，默認是252
    :return: 年化波動率
    """
    prices = np.asarray(prices)
    # 計算對數收益率
    log_returns = np.log(prices[1:] / prices[:-1])

    # 計算對數收益率的標準差
    std_dev = np.std(log_returns[-window:])

    # 年化波動率
    annualized_volatility = std_dev * np.sqrt(trading_days)
    
    return annualized_volatility

def cal_tech(id):

    path = os.path.join(os.path.abspath(os.path.dirname(os.getcwd())), 'history_data', 'us', 'stock_price')
    # folder_list = ['2330', '2454', '2317', '2382', '2308', '3711']


    stock_tech = pd.read_csv(os.path.join(path , id + '.csv'))

    # # 將stock_tech的值都換成float
    # stock_tech['Open'] = stock_tech['Open'].map(lambda x: float(x.replace(',', '')))
    # stock_tech['Close'] = stock_tech['Close'].map(lambda x: float(x.replace(',', '')))
    # stock_tech['High'] = stock_tech['High'].map(lambda x: float(x.replace(',', '')))
    # stock_tech['Low'] = stock_tech['Low'].map(lambda x: float(x.replace(',', '')))


    # 計算移動均線 (SMA)
    stock_tech['MA5'] = stock_tech['Close'].rolling(window=5).mean()
    stock_tech['MA10'] = stock_tech['Close'].rolling(window=10).mean()
    stock_tech['MA20'] = stock_tech['Close'].rolling(window=20).mean()

    for i in range(1, 257):
        # 計算指數移動平均線 (EMA)
        stock_tech['EMA' + str(i)] = stock_tech['Close'].ewm(span=i, adjust=False).mean()
    

    # # 計算MACD和DIF
    # stock_tech['DIF'] = stock_tech['EMA12'] - stock_tech['EMA26']
    # stock_tech['MACD'] = stock_tech['DIF'].ewm(span=9, adjust=False).mean()


    # # 計算RSI
    # def calculate_rsi(series, period=14):
    #     delta = series.diff(1)
    #     gain = (delta.where(delta > 0, 0)).rolling(window=period).mean()
    #     loss = (-delta.where(delta < 0, 0)).rolling(window=period).mean()
    #     rs = gain / loss
    #     rsi = 100 - (100 / (1 + rs))
    #     return rsi

    # stock_tech['RSI'] = calculate_rsi(stock_tech['Close'])

    # # 計算KDJ
    # low_list = stock_tech['Low'].rolling(9, min_periods=9).min()
    # low_list.fillna(value=stock_tech['Low'].expanding().min(), inplace=True)
    # high_list = stock_tech['High'].rolling(9, min_periods=9).max()
    # high_list.fillna(value=stock_tech['High'].expanding().max(), inplace=True)
    # rsv = (stock_tech['Close'] - low_list) / (high_list - low_list) * 100

    # stock_tech['Volatility'] = stock_tech['Close'].rolling(window=252).apply(historical_volatility, raw=True)

    # # 初始值設定
    # stock_tech['K'] = 50
    # stock_tech['D'] = 50

    # for i in range(1, len(stock_tech)):
    #     stock_tech.loc[stock_tech.index[i], 'K'] = (2/3) * stock_tech.loc[stock_tech.index[i-1], 'K'] + (1/3) * rsv[i]
    #     stock_tech.loc[stock_tech.index[i], 'D'] = (2/3) * stock_tech.loc[stock_tech.index[i-1], 'D'] + (1/3) * stock_tech.loc[stock_tech.index[i], 'K']


    # 顯示DataFrame
    stock_tech.to_csv(os.path.join(path , id + 'tech.csv'), encoding='utf-8', index=False)

In [11]:
data = [
    ["AMGN", "Amgen", "us"],
    ["AMZN", "Amazon inc.", "us"],
    ["AAPL", "Apple Inc.", "us"],
    ["AXP", "American Express", "us"],
    ["BA", "Boeing", "us"],
    ["CAT", "Caterpillar", "us"],
    ["CVX", "Chevron", "us"],
    ["CSCO", "Cisco", "us"],
    ["KO", "Coca-Cola", "us"],
    ["CRM", "Salesforce", "us"],    # 10
    ["DIS", "Disney", "us"],
    ["GS", "Goldman Sachs", "us"],
    ["HD", "Home Depot", "us"],
    ["HON", "Honeywell", "us"],
    ["IBM", "IBM", "us"],
    ["JNJ", "Johnson & Johnson", "us"],
    ["JPM", "JPMorgan Chase", "us"],
    ["MCD", "McDonald's", "us"],
    ["MRK", "Merck", "us"],
    ["MSFT", "Microsoft", "us"],    # 20
    ["MMM", "3M", "us"],
    ["NKE", "Nike", "us"],
    ["NVDA", "Nvidia", "us"],
    ["PG", "Procter & Gamble", "us"],
    ["SHW", "Sherwin-Williams", "us"],
    ["TRV", "Travelers", "us"],
    ["UNH", "UnitedHealth Group", "us"],
    ["VZ", "Verizon", "us"],
    ["V", "Visa", "us"],
    ["WMT", "Walmart", "us"],   # 30
    
    ["INTC", "Intel", "us"],
    ["DOW", "Dow Inc.", "us"],
    ["WBA", "Walgreens Boots Alliance", "us"],
]

for d in data:
    # 指定檔案路徑
    stock_id = d[0]
    input_file = os.path.join(os.path.abspath(os.path.dirname(os.getcwd())), 'history_data', 'us', 'stock_price_raw', stock_id + '.csv')
    output_file = os.path.join(os.path.abspath(os.path.dirname(os.getcwd())), 'history_data', 'us', 'stock_price', stock_id + '.csv')
    convert_file(input_file, output_file)
    print(f"檔案已轉換並儲存至 {output_file}")
    cal_tech(stock_id)
    print('save ' + stock_id + 'tech.csv')

檔案已轉換並儲存至 /Users/yanyifu/Documents/_Coding/LLM Predict Stock/history_data/us/stock_price/AMGN.csv


/var/folders/xc/rjj8m66j6xs5pgcp1hc7g2kr0000gn/T/ipykernel_13972/2001430637.py:52: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  stock_tech['EMA' + str(i)] = stock_tech['Close'].ewm(span=i, adjust=False).mean()
/var/folders/xc/rjj8m66j6xs5pgcp1hc7g2kr0000gn/T/ipykernel_13972/2001430637.py:52: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  stock_tech['EMA' + str(i)] = stock_tech['Close'].ewm(span=i, adjust=False).mean()
/var/folders/xc/rjj8m66j6xs5pgcp1hc7g2kr0000gn/T/ipykernel_13972/2001430637.py:52: PerformanceWarning: DataFrame

save AMGNtech.csv
檔案已轉換並儲存至 /Users/yanyifu/Documents/_Coding/LLM Predict Stock/history_data/us/stock_price/AMZN.csv


/var/folders/xc/rjj8m66j6xs5pgcp1hc7g2kr0000gn/T/ipykernel_13972/2001430637.py:52: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  stock_tech['EMA' + str(i)] = stock_tech['Close'].ewm(span=i, adjust=False).mean()
/var/folders/xc/rjj8m66j6xs5pgcp1hc7g2kr0000gn/T/ipykernel_13972/2001430637.py:52: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  stock_tech['EMA' + str(i)] = stock_tech['Close'].ewm(span=i, adjust=False).mean()
/var/folders/xc/rjj8m66j6xs5pgcp1hc7g2kr0000gn/T/ipykernel_13972/2001430637.py:52: PerformanceWarning: DataFrame

save AMZNtech.csv
檔案已轉換並儲存至 /Users/yanyifu/Documents/_Coding/LLM Predict Stock/history_data/us/stock_price/AAPL.csv


...

save INTCtech.csv
檔案已轉換並儲存至 /Users/yanyifu/Documents/_Coding/LLM Predict Stock/history_data/us/stock_price/DOW.csv


/var/folders/xc/rjj8m66j6xs5pgcp1hc7g2kr0000gn/T/ipykernel_13972/2001430637.py:52: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  stock_tech['EMA' + str(i)] = stock_tech['Close'].ewm(span=i, adjust=False).mean()
/var/folders/xc/rjj8m66j6xs5pgcp1hc7g2kr0000gn/T/ipykernel_13972/2001430637.py:52: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  stock_tech['EMA' + str(i)] = stock_tech['Close'].ewm(span=i, adjust=False).mean()
/var/folders/xc/rjj8m66j6xs5pgcp1hc7g2kr0000gn/T/ipykernel_13972/2001430637.py:52: PerformanceWarning: DataFrame

save DOWtech.csv
檔案已轉換並儲存至 /Users/yanyifu/Documents/_Coding/LLM Predict Stock/history_data/us/stock_price/WBA.csv


/var/folders/xc/rjj8m66j6xs5pgcp1hc7g2kr0000gn/T/ipykernel_13972/2001430637.py:52: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  stock_tech['EMA' + str(i)] = stock_tech['Close'].ewm(span=i, adjust=False).mean()
/var/folders/xc/rjj8m66j6xs5pgcp1hc7g2kr0000gn/T/ipykernel_13972/2001430637.py:52: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  stock_tech['EMA' + str(i)] = stock_tech['Close'].ewm(span=i, adjust=False).mean()
/var/folders/xc/rjj8m66j6xs5pgcp1hc7g2kr0000gn/T/ipykernel_13972/2001430637.py:52: PerformanceWarning: DataFrame

save WBAtech.csv
